# Zero to Verifiable — 60-minute Colab
### Teaching a small model to solve Countdown: SFT shows one solution, GRPO rewards any

**The puzzle:** combine all the given numbers, each exactly once, with `+ − × ÷` to hit a target. `14 18 3 → 68` is solved by `3×18=54`, then `14+54=68`. Finding a solution is a search; checking one takes a few lines of code. That gap is what reinforcement learning with a verifier exploits.

SFT shows the model **one** solution per puzzle. Most puzzles have several (the 4-number ones have a median of 18), and GRPO rewards **any** correct one the model finds by sampling. We measure how much room that leaves before RL starts: how often at least one of 8 sampled answers is right, compared with the single greedy answer.

| | |
|---|---|
| **Model** | `Qwen/Qwen2.5-0.5B-Instruct` (revision-pinned), LoRA r=16 on 16-bit weights, free T4 |
| **GPU budget** | ~4 min SFT + ~12 min GRPO + ~3 min evaluation ≈ **19 min of GPU** |
| **What you see live** | `train/loss`, `eval/exact_match_sft`, `train/reward`, `train/frac_reward_zero_std`, `eval/exact_match_rl` |
| **Data** | every puzzle is **generated inside this notebook**. Nothing to download or share except this notebook; cell 1 fetches the packages and model weights. |

**Run it:** `Runtime ▸ Change runtime type ▸ T4 GPU`, then run top to bottom. Cells marked ⏱ take minutes: start them, then talk.

In [ ]:
# 1 · Setup. Pinned to the versions validated on Colab; torch stays as Colab ships it
#     (replacing Colab's torch breaks its CUDA build).
%pip install -q transformers==5.22.1 trl==1.18.1 peft==0.21.3

import os, re, math, time, random
import numpy as np, pandas as pd
import torch, transformers, trl, peft

assert torch.cuda.is_available(), "No GPU -> Runtime > Change runtime type > T4 GPU"
cap  = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
BF16  = cap[0] >= 8      # real bf16 needs Ampere+; torch.cuda.is_bf16_supported() also says True on a T4 (emulated, slow)
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(f"{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}  {vram:.1f} GiB  bf16={BF16}")
print({"torch": torch.__version__, "transformers": transformers.__version__, "trl": trl.__version__, "peft": peft.__version__})

RUNS = "/content/runs"; OUT = "/content/ckpt"      # everything under RUNS shows up in TensorBoard
os.makedirs(RUNS, exist_ok=True); os.makedirs(OUT, exist_ok=True)
SEED = 3407; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

from huggingface_hub import snapshot_download
MODEL_ID  = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REV = "7ae557604adf67be50417f59c2c2f167def9a775"   # pinned: an upstream template change can't change the task mid-workshop
snapshot_download(MODEL_ID, revision=MODEL_REV)          # ~1 GB of weights, fetched now so section 5 loads from cache
print("model files cached")

In [ ]:
# 2 · TensorBoard — the live window into loss, reward and accuracy. It reads every run under /content/runs.
#     It stays empty until training logs its first step; use the panel's refresh icon if it lags.
%load_ext tensorboard
%tensorboard --logdir /content/runs

## 3 · Data — generated here, checked by a few lines of code

**Rules.** Use every number exactly once; each step combines two available numbers with `+ − × ÷`; every intermediate result must be a positive whole number. The answer is written one step per line, `a op b = c`, ending at the target.

**Generator.** Pick 3 or 4 numbers from 1–20, combine them in a random legal order, and keep the result as the target if it lands between 10 and 100. That random path is the one worked solution SFT gets to see. Every puzzle is unique across splits:

| split | 3 numbers | 4 numbers | used for |
|---|---|---|---|
| SFT | 5,000 | 11,000 | worked examples |
| RL | 1,000 | 3,000 | GRPO prompts, scored only by the checker |
| dev | 32 | 32 | the live accuracy curves |
| test | 150 | 150 | the final table, touched once |
| bigger | 30 | 30 | numbers up to 50 instead of 20 |

**The floor.** A small solver lists every legal way to combine a puzzle's numbers. Picking one of them at random hits the target only a few percent of the time; that's the number a model has to beat.

In [ ]:
def apply(a, op, b):
    "Countdown rules: the result must be a positive whole number."
    c = {"+": a + b, "-": a - b, "*": a * b, "/": a // b if b and a % b == 0 else None}[op]
    return c if c is not None and c > 0 else None

def random_solution(nums, rng):
    "Combine the numbers in a random legal order; returns the steps and the final value."
    pool, steps = list(nums), []
    while len(pool) > 1:
        i, j = rng.sample(range(len(pool)), 2); op = rng.choice("+-*/")
        c = apply(pool[i], op, pool[j])
        if c is None: return None
        steps.append(f"{pool[i]}{op}{pool[j]}={c}")
        pool = [x for k, x in enumerate(pool) if k not in (i, j)] + [c]
    return steps, pool[0]

def make_puzzle(rng, n, hi):
    while True:
        nums = [rng.randint(1, hi) for _ in range(n)]
        sol = random_solution(nums, rng)
        if sol and 10 <= sol[1] <= 100 and sol[1] not in nums:
            return {"nums": nums, "target": sol[1], "steps": sol[0]}

SPLITS = {"test": (150, 150, 20), "dev": (32, 32, 20), "rl": (1000, 3000, 20), "train": (5000, 11000, 20),
          "bigger": (30, 30, 50)}                    # (3-number puzzles, 4-number puzzles, largest number)
rng, used = random.Random(2026), set()
def fresh(n, hi):
    while True:
        p = make_puzzle(rng, n, hi); key = (tuple(sorted(p["nums"])), p["target"])
        if key not in used: used.add(key); return p
PUZZLES = {s: [fresh(3, hi) for _ in range(n3)] + [fresh(4, hi) for _ in range(n4)] for s, (n3, n4, hi) in SPLITS.items()}

def derivations(pool, steps=()):
    "Every legal way to combine all the numbers: yields (final value, steps)."
    if len(pool) == 1: yield pool[0], steps; return
    for i in range(len(pool)):
        for j in range(len(pool)):
            for op in "+-*/":
                if i == j or (op in "+*" and i > j): continue      # a+b and b+a count once
                c = apply(pool[i], op, pool[j])
                if c is None: continue
                rest = [x for k, x in enumerate(pool) if k not in (i, j)]
                yield from derivations(rest + [c], steps + (f"{pool[i]}{op}{pool[j]}={c}",))

STEP = re.compile(r"(\d+)\s*([-+*/x×÷])\s*(\d+)\s*=\s*(-?\d+)")
def check(text, nums):
    "Replay the steps. Legal = every step is true arithmetic on numbers still available, and all numbers get used."
    pool, steps = list(nums), STEP.findall(text or "")
    if not steps: return False, None
    for a, op, b, c in steps:
        a, b, c, op = int(a), int(b), int(c), {"x": "*", "×": "*", "÷": "/"}.get(op, op)
        if a not in pool: return False, None
        pool.remove(a)
        if b not in pool: return False, None
        pool.remove(b)
        if apply(a, op, b) != c: return False, None
        pool.append(c)
    return len(pool) == 1, (pool[0] if len(pool) == 1 else None)

SYS = ("You solve Countdown puzzles: combine all the given numbers, each exactly once, using + - * / to reach the "
       "target. Every intermediate result must be a positive whole number. Write one step per line, like 4*5=20, "
       "and stop when you reach the target.")

def make_row(split, p):
    r = {"split": split, "n": len(p["nums"]), "nums": p["nums"], "target": p["target"],
         "prompt": f"Numbers: {' '.join(map(str, p['nums']))}\nTarget: {p['target']}", "answer": "\n".join(p["steps"])}
    if split in ("test", "bigger", "seen", "rl-seen"):          # the floor: chance that a random legal path hits
        finals = [v for v, _ in derivations(p["nums"])]
        r["p_random"], r["n_solutions"] = sum(v == p["target"] for v in finals) / len(finals), finals.count(p["target"])
    return r

DATA = {s: [make_row(s, p) for p in ps] for s, ps in PUZZLES.items()}
TRAIN, RL_ROWS, DEV, TEST, BIGGER = DATA["train"], DATA["rl"], DATA["dev"], DATA["test"], DATA["bigger"]
SEEN    = [make_row("seen", p) for p in random.Random(5).sample(PUZZLES["train"], 40)]
RL_SEEN = [make_row("rl-seen", p) for p in random.Random(6).sample(PUZZLES["rl"], 40)]
print("puzzles -> SFT", len(TRAIN), "| RL", len(RL_ROWS), "| dev", len(DEV), "| test", len(TEST), "| bigger", len(BIGGER))

assert all(check(r["answer"], r["nums"]) == (True, r["target"]) for r in TRAIN + RL_ROWS + TEST), \
       "the checker must accept every worked solution"
print("the checker accepts every worked solution")
for n in (3, 4):
    t = [r for r in TEST if r["n"] == n]
    print(f"test, {n} numbers: a random legal path hits {100*np.mean([r['p_random'] for r in t]):.1f}% | "
          f"median {int(np.median([r['n_solutions'] for r in t]))} solutions per puzzle")
print("\nexample prompt:\n" + TRAIN[1]["prompt"] + "\n\nits worked solution:\n" + TRAIN[1]["answer"])

## 4 · Turn a puzzle into a training example

One system turn, one user turn, one assistant answer. The label mask is the teaching point: **everything before the assistant's answer is `-100`**, so the loss only sees the steps it must produce. Training and generation call the *same* `render()`.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"        # generation needs left padding; SFT batches are padded by collate() in section 7
print("pad token:", tok.pad_token, "| eos:", tok.eos_token)

def msgs(r):
    return [{"role": "system", "content": SYS}, {"role": "user", "content": r["prompt"]}]

def render(r, answer=None):
    "The one place a row becomes text. Training passes the answer; generation leaves the assistant turn open."
    m = msgs(r) + ([{"role": "assistant", "content": answer}] if answer is not None else [])
    return tok.apply_chat_template(m, tokenize=False, add_generation_prompt=answer is None)

def ids(text): return tok(text, add_special_tokens=False)["input_ids"]

def encode(r):
    pre, full = ids(render(r)), ids(render(r, r["answer"]))
    assert full[:len(pre)] == pre, "chat-template prefix mismatch: the prompt tokens change once the answer is appended"
    return {"input_ids": full, "labels": [-100]*len(pre) + full[len(pre):], "attention_mask": [1]*len(full)}

lens = [(len(e["input_ids"]), sum(t != -100 for t in e["labels"])) for e in map(encode, random.sample(TRAIN, 200))]
print(f"tokens per row: mean {np.mean([a for a, b in lens]):.0f} | supervised (the steps): mean {np.mean([b for a, b in lens]):.0f},"
      f" max {max(b for a, b in lens)}")

## 5 · Base model + LoRA

Qwen2.5-0.5B in 16-bit is ~1 GB, a third of the 1.5B model: every step, and especially every sampled answer in GRPO, is about three times cheaper. A rank-16 LoRA adapter trains ~9 M parameters; the base weights stay frozen.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REV, dtype=DTYPE, device_map={"": 0})
model.config.use_cache = False                    # training; generate() turns the KV cache back on
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM"))
model.print_trainable_parameters()
print("VRAM after load: %.2f GiB" % (torch.cuda.max_memory_allocated()/2**30))

## 6 · The "before": measure it, don't assume it

Two measurements, every time:
- **greedy**: the model's single best answer. This is what "accuracy" usually means.
- **8 samples at temperature 1**: how GRPO sees the model. The share of puzzles where **any of 8** samples is right is the room RL has to work with, because RL can only reinforce answers the model sometimes produces.

`legal steps` counts answers whose arithmetic is all true and that use every number, whether or not they hit the target.

In [ ]:
GEN_KW = dict(max_new_tokens=96, repetition_penalty=1.0, use_cache=True, pad_token_id=tok.pad_token_id,
              eos_token_id=[tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")])
# Passed explicitly: Qwen's shipped generation_config samples at T=0.7 with top-p/top-k and adds repetition_penalty=1.1.

@torch.inference_mode()
def generate(items, k=1):
    "Greedy when k == 1; otherwise k samples per puzzle at temperature 1, the way GRPO explores."
    was_training = model.training; model.eval()
    mode = dict(do_sample=False) if k == 1 else dict(do_sample=True, temperature=1.0, top_p=1.0, top_k=0, num_return_sequences=k)
    batch, outs = 64 if k == 1 else 32, []
    for i in range(0, len(items), batch):
        chunk = items[i:i+batch]
        enc = tok([render(r) for r in chunk], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.autocast("cuda", dtype=DTYPE):
            gen = model.generate(**enc, **GEN_KW, **mode)
        texts = [s.strip() for s in tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
        outs += texts if k == 1 else [texts[j*k:(j+1)*k] for j in range(len(chunk))]
    if was_training: model.train()
    return outs

def solved(r, text):
    legal, final = check(text, r["nums"])
    return int(legal and final == r["target"])

def score(items, outs):
    return [solved(r, o) for r, o in zip(items, outs)], [int(check(o, r["nums"])[0]) for r, o in zip(items, outs)]

EVAL = TEST + BIGGER + SEEN + RL_SEEN
def report(name, outs):
    ex, lg = score(EVAL, outs)
    df = pd.DataFrame({"split": [r["split"] for r in EVAL], "n": [r["n"] for r in EVAL], "exact": ex, "legal": lg})
    for split, g in df.groupby("split", sort=False):
        extra = ("   (" + ", ".join(f"{n} numbers {100*h.exact.mean():.1f}%" for n, h in g.groupby("n")) + ")") if split == "test" else ""
        print(f"{name:<7} {split:<8} greedy solved {100*g.exact.mean():5.1f}%   legal steps {100*g.legal.mean():5.1f}%   n={len(g)}{extra}")

def sampled(name, k=8):
    hits = np.array([[solved(r, o) for o in outs] for r, outs in zip(TEST, generate(TEST, k=k))])
    print(f"{name:<7} test, {k} samples each: average {100*hits.mean():5.1f}%   any of {k} right {100*hits.max(1).mean():5.1f}%")
    return hits

def show(outs, k=4):
    for r, o in zip(EVAL[:k], outs[:k]):
        print(f"{'✓' if solved(r, o) else '✗'} {r['prompt'].replace(chr(10), ' | ')}  ->  {o[:90]!r}")

with model.disable_adapter():                     # base weights only
    BASE_OUT = generate(EVAL); report("base", BASE_OUT); BASE_HITS = sampled("base")
show(BASE_OUT)

## 7 · SFT ⏱ ~4 min: start it, then talk

One worked solution per puzzle, in batches of 32. We time one real optimizer step first, then derive `max_steps` from a **time budget**, so a slower GPU trains less instead of overrunning the session. A wall-clock callback is the backstop. SFT is kept short on purpose: the point is to teach the format and a decent instinct, and leave room for RL. TensorBoard gets `train/loss` from the trainer and `eval/exact_match_sft` (greedy, on the 64 dev puzzles) every 25 steps.

In [ ]:
from torch.utils.data import Dataset as TorchDS
from torch.utils.tensorboard import SummaryWriter
from transformers import Trainer, TrainingArguments, TrainerCallback

class DS(TorchDS):
    def __init__(self, rows): self.rows = rows
    def __len__(self):  return len(self.rows)
    def __getitem__(self, i): return encode(self.rows[i])

def collate(batch):
    mx = max(len(b["input_ids"]) for b in batch)
    ids_ = torch.full((len(batch), mx), tok.pad_token_id, dtype=torch.long)
    lab  = torch.full((len(batch), mx), -100, dtype=torch.long)
    am   = torch.zeros((len(batch), mx), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids_[i, :n] = torch.tensor(b["input_ids"])
        lab[i, :n]  = torch.tensor(b["labels"])
        am[i, :n]   = 1
    return {"input_ids": ids_, "labels": lab, "attention_mask": am}

WRITER = SummaryWriter(f"{RUNS}/eval")

class TBAcc(TrainerCallback):
    "Greedy solve rate on the dev puzzles at the start and every `every` steps: the accuracy curve, not just the loss."
    def __init__(self, key, items, every): self.key, self.items, self.every = key, items, every
    def log(self, step):
        acc = 100 * np.mean(score(self.items, generate(self.items))[0])
        WRITER.add_scalar(self.key, acc, step); WRITER.flush()
        print(f"  [step {step}] {self.key} = {acc:.1f}%")
    def on_train_begin(self, args, state, control, **kw): self.log(state.global_step)
    def on_step_end(self, args, state, control, **kw):
        if state.global_step % self.every == 0: self.log(state.global_step)

class StopAt(TrainerCallback):
    "Hard wall-clock stop: the agenda is the budget."
    def __init__(self, seconds): self.seconds, self.t0 = seconds, None
    def on_train_begin(self, args, state, control, **kw): self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        if time.time() - self.t0 > self.seconds:
            control.should_training_stop = True
            print(f"  wall-clock stop at step {state.global_step}")

# --- timing probe: one optimizer step of BS rows ---
SFT_MIN, BS = 4, 32
def probe_step():
    b = collate([encode(r) for r in random.sample(TRAIN, BS)])
    with torch.autocast("cuda", dtype=DTYPE):
        model(**{k: v.cuda() for k, v in b.items()}).loss.backward()
model.train()
probe_step(); torch.cuda.synchronize()            # warm-up, not timed
t0 = time.time(); probe_step(); torch.cuda.synchronize(); S_PER_STEP = time.time() - t0
model.zero_grad(set_to_none=True)

steps_per_epoch = math.ceil(len(TRAIN) / BS)
MAX_STEPS = min(2*steps_per_epoch, max(50, int(0.85*SFT_MIN*60 / S_PER_STEP)))
print(f"{S_PER_STEP:.2f} s per optimizer step ({BS} puzzles) | 1 epoch = {steps_per_epoch} steps"
      f" -> max_steps {MAX_STEPS} (~{MAX_STEPS*S_PER_STEP/60:.1f} min, {MAX_STEPS*BS:,} puzzles)")

args = TrainingArguments(
    output_dir=f"{RUNS}/sft", per_device_train_batch_size=BS, max_steps=MAX_STEPS,
    learning_rate=2e-4, lr_scheduler_type="linear", warmup_steps=20, logging_steps=5,
    report_to=["tensorboard"], bf16=BF16, fp16=not BF16, save_strategy="no", remove_unused_columns=False, seed=SEED)
os.environ["TENSORBOARD_LOGGING_DIR"] = f"{RUNS}/sft"
trainer = Trainer(model=model, args=args, train_dataset=DS(TRAIN), data_collator=collate,
                  callbacks=[TBAcc("eval/exact_match_sft", DEV, every=25), StopAt(SFT_MIN*60)])
trainer.train()
model.save_pretrained(f"{OUT}/adapter_sft"); tok.save_pretrained(f"{OUT}/adapter_sft")
print("saved:", f"{OUT}/adapter_sft")

## 8 · After SFT: how much room is left for RL?

Same puzzles, same renderer. The line to read is the last one: if **any of 8** is well above **greedy**, the model already finds solutions it doesn't commit to, and that gap is what GRPO can turn into greedy accuracy. If the two are close, RL has little to work with, and that is worth saying out loud.

In [ ]:
SFT_OUT = generate(EVAL); report("SFT", SFT_OUT); SFT_HITS = sampled("SFT")
show(SFT_OUT)
print(f"\nroom for RL: greedy {100*np.mean(score(TEST, SFT_OUT[:len(TEST)])[0]):.1f}% vs any of 8 {100*SFT_HITS.max(1).mean():.1f}%")

## 9 · RL: GRPO against the checker

GRPO samples **8 answers per puzzle** from four thousand RL puzzles SFT never saw, and the checker scores each one:

- **1.0** for a legal answer that hits the target, *whichever* solution it is;
- **up to 0.5** for a legal answer that misses, shrinking by 10% for every unit it is off (0.45 at one off, 0.17 at ten off);
- **0** for anything else: false arithmetic, a number used twice or left out, or no steps at all.

That makes the reward dense without being cheatable: writing `13+20=98` to "hit" the target earns nothing. GRPO compares each answer with the other 7 for the same puzzle; if all 8 score the same, the puzzle teaches nothing, and `train/frac_reward_zero_std` in TensorBoard is the share of such puzzles.

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

def reward_value(text, nums, target):
    legal, final = check(text, nums)
    if not legal: return 0.0
    return 1.0 if final == target else 0.5 * 0.9 ** abs(final - target)

def reward_countdown(completions, nums, target, **kw):
    texts = [c if isinstance(c, str) else c[-1]["content"] for c in completions]   # chat prompts -> message lists
    return [reward_value(t, ns, tg) for t, ns, tg in zip(texts, nums, target)]

r0 = next(r for r in RL_ROWS if r["n"] == 4); misses = [d for d in derivations(r0["nums"]) if d[0] != r0["target"]]
def miss_by(off):
    "A legal path that misses the target by about `off`."
    v, steps = min(misses, key=lambda d: abs(abs(d[0] - r0["target"]) - off))
    return f"legal, misses by {abs(v - r0['target'])}", "\n".join(steps)
first = r0["answer"].splitlines(); lhs, rhs = first[0].split("=")
print("puzzle:", r0["prompt"].replace("\n", " | "))
for label, txt in [("its worked solution", r0["answer"]), miss_by(1), miss_by(10),
                   ("false arithmetic", "\n".join([f"{lhs}={int(rhs) + 1}"] + first[1:])),
                   ("a number left out", "\n".join(first[:-1])), ("no steps", "I am not sure")]:
    print(f"  {label:<22} -> reward {reward_value(txt, r0['nums'], r0['target']):.3f}   {txt.replace(chr(10), ' ; ')[:60]}")

RL_DS = Dataset.from_list([{"prompt": msgs(r), "nums": r["nums"], "target": r["target"]} for r in RL_ROWS])
print(f"RL puzzles: {len(RL_DS)}")

RL_MIN, RL_LR = 12, 2e-5
grpo_args = GRPOConfig(
    output_dir=f"{RUNS}/grpo", per_device_train_batch_size=32, gradient_accumulation_steps=2,
    num_generations=8, max_completion_length=64, temperature=1.0, beta=0.0, scale_rewards="group",
    learning_rate=RL_LR, lr_scheduler_type="constant_with_warmup", warmup_steps=10, max_grad_norm=0.1,
    max_steps=1000, logging_steps=1, report_to=["tensorboard"], bf16=BF16, fp16=not BF16,
    gradient_checkpointing=False, save_strategy="no", seed=SEED)
os.environ["TENSORBOARD_LOGGING_DIR"] = f"{RUNS}/grpo"
grpo = GRPOTrainer(model=model, reward_funcs=reward_countdown, args=grpo_args, train_dataset=RL_DS,
                   processing_class=tok, callbacks=[TBAcc("eval/exact_match_rl", DEV, every=25), StopAt(RL_MIN*60)])
print(f"ready: 8 puzzles x 8 samples per step, lr {RL_LR}, stops at {RL_MIN} min")

## 10 · Train the RL stage ⏱ ~12 min: start it, then talk

What to watch: `train/reward` should climb, and `train/frac_reward_zero_std` should stay well below 1. `train/completions/mean_length` should stay near the length of a worked solution: shrinking can mean the model has found a shortcut the checker allows, which is worth a look at the samples.

In [ ]:
grpo.train()
model.save_pretrained(f"{OUT}/adapter_rl"); tok.save_pretrained(f"{OUT}/adapter_rl")
RL_OUT = generate(EVAL); report("SFT→RL", RL_OUT); RL_HITS = sampled("SFT→RL")
show(RL_OUT)

## 11 · The result: four arms, one table

One model and one renderer; only the adapter changes: **base** (adapter off) → **SFT** → **SFT→RL** (GRPO continued the SFT adapter). The first row is the floor: picking a random legal path through each puzzle.

Puzzles are independent, so the 95% intervals resample the 300 test puzzles. `bigger` uses numbers up to 50 (never trained on); `seen` and `rl puzzles` are 40-puzzle spot checks from the SFT and RL sets.

Read the last two columns together. **Greedy** and **avg of 8** measure how reliably the model solves a puzzle; **any of 8** measures how many puzzles it can solve at all when it gets several tries. RL usually raises the first two and can lower the third: it turns attempts that sometimes work into answers the model commits to, and it explores less along the way.

In [ ]:
import matplotlib.pyplot as plt

def boot(x, B=10_000, seed=0):
    "Mean and 95% bootstrap interval over puzzles."
    x = np.asarray(x, float); m = x[np.random.default_rng(seed).integers(0, len(x), (B, len(x)))].mean(1)
    return 100*x.mean(), 100*np.percentile(m, 2.5), 100*np.percentile(m, 97.5)

nT = len(TEST)
p_rand = np.array([r["p_random"] for r in EVAL])
ARMS = {"random legal path": dict(exact=p_rand, legal=np.ones(len(EVAL)), sampled=p_rand[:nT], any8=1 - (1 - p_rand[:nT])**8)}
for arm, outs, hits in [("base", BASE_OUT, BASE_HITS), ("SFT", SFT_OUT, SFT_HITS), ("SFT→RL", RL_OUT, RL_HITS)]:
    ex, lg = score(EVAL, outs)
    ARMS[arm] = dict(exact=np.array(ex, float), legal=np.array(lg, float), sampled=hits.mean(1), any8=hits.max(1).astype(float))

split = np.array([r["split"] for r in EVAL]); n = np.array([r["n"] for r in EVAL])
rows = {}
for arm, a in ARMS.items():
    e, lo, hi = boot(a["exact"][:nT])
    rows[arm] = {"greedy": e, "ci_lo": lo, "ci_hi": hi, "legal": 100*a["legal"][:nT].mean(),
                 "3 numbers": 100*a["exact"][(split == "test") & (n == 3)].mean(),
                 "4 numbers": 100*a["exact"][(split == "test") & (n == 4)].mean(),
                 "bigger": 100*a["exact"][split == "bigger"].mean(), "seen": 100*a["exact"][split == "seen"].mean(),
                 "rl puzzles": 100*a["exact"][split == "rl-seen"].mean(),
                 "avg of 8": 100*a["sampled"].mean(), "any of 8": 100*a["any8"].mean()}
tab = pd.DataFrame(rows).T
print("test = 300 new puzzles (150 with 3 numbers, 150 with 4) | bigger = numbers up to 50\n")
print(tab.round(1).to_string())
pd.DataFrame({arm: a["exact"] for arm, a in ARMS.items()}).assign(split=split).to_csv(f"{OUT}/results.csv", index=False)

print()
DELTAS = {}
for a, b, what in [("SFT", "base", "exact"), ("SFT→RL", "SFT", "exact"), ("SFT→RL", "SFT", "sampled"), ("SFT→RL", "SFT", "any8")]:
    d, lo, hi = DELTAS[a, b, what] = boot(ARMS[a][what][:nT] - ARMS[b][what][:nT])     # paired: same 300 puzzles
    label = {"exact": "greedy", "sampled": "avg of 8", "any8": "any of 8"}[what]
    print(f"{a:>6} − {b:<5} {label:<8} {d:+6.1f} pp   95% CI [{lo:+.1f}, {hi:+.1f}]   "
          f"{'distinguishable' if lo > 0 or hi < 0 else 'NOT distinguishable'} from zero")
s, rl = ARMS["SFT"]["exact"][:nT], ARMS["SFT→RL"]["exact"][:nT]
print(f"RL vs SFT, puzzle by puzzle (greedy): {int(((rl == 1) & (s == 0)).sum())} newly solved, {int(((rl == 0) & (s == 1)).sum())} lost")

d, lo, hi = DELTAS["SFT→RL", "SFT", "exact"]
a8 = DELTAS["SFT→RL", "SFT", "any8"]
explore = ("it commits more and explores less" if a8[2] < 0 else "it also finds more puzzles it can solve" if a8[1] > 0
           else "the change is not distinguishable from zero")
print("\nThe sentence these numbers allow:")
print(f'  "On 300 new Countdown puzzles, a random legal path hits the target {tab.loc["random legal path", "greedy"]:.0f}% of the time '
      f'and the base model solved {tab.loc["base", "greedy"]:.0f}%. After SFT on one worked solution per puzzle it solved '
      f'{tab.loc["SFT", "greedy"]:.0f}%, and at least one of 8 samples was right on {tab.loc["SFT", "any of 8"]:.0f}%. '
      f'GRPO, rewarded for any correct solution, changed greedy accuracy by {d:+.1f} pp (95% CI {lo:+.1f} to {hi:+.1f}), '
      f'which is {"" if lo > 0 or hi < 0 else "not "}distinguishable from zero. Meanwhile "any of 8 right" moved by '
      f'{a8[0]:+.1f} pp (95% CI {a8[1]:+.1f} to {a8[2]:+.1f}): {explore}."')

fig, ax = plt.subplots(figsize=(5.6, 3.0), dpi=150)
x = np.arange(len(tab)); w = 0.38
ax.bar(x - w/2, tab["greedy"], w, color="#2b6cb0", label="greedy",
       yerr=[tab["greedy"] - tab["ci_lo"], tab["ci_hi"] - tab["greedy"]], capsize=3, ecolor="#222")
ax.bar(x + w/2, tab["any of 8"], w, color="#a0aec0", label="any of 8 samples")
ax.set_xticks(x); ax.set_xticklabels(tab.index, fontsize=7); ax.set_ylim(0, 100); ax.set_ylabel("puzzles solved, %")
ax.legend(fontsize=7, loc="upper left"); ax.set_title("300 new puzzles, 95% CI on greedy", fontsize=8)
plt.tight_layout(); plt.show()

## 12 · What did RL change in the answers?

The checker only cares whether an answer is legal and hits the target, so RL is free to change *how* the model solves puzzles. The table compares the operators in the worked solutions SFT imitated with the operators in the models' legal answers on the test puzzles. If RL leans on `+` and `−`, it has learned that easy moves are safer bets: they are legal more often, and near-misses still earn partial reward. Is that a skill or a shortcut?

In [ ]:
from collections import Counter
CANON = {"x": "*", "×": "*", "÷": "/"}

def op_profile(texts):
    "Operator mix over the legal answers: which moves does the model reach for?"
    legal = [o for r, o in zip(TEST, texts) if check(o, r["nums"])[0]]
    used = [[CANON.get(op, op) for _, op, _, _ in STEP.findall(o)] for o in legal]
    ops = Counter(op for u in used for op in u); total = sum(ops.values()) or 1
    return {**{f"{op} share": 100 * ops[op] / total for op in "+-*/"},
            "only + and −": 100 * np.mean([set(u) <= {"+", "-"} for u in used]) if used else 0.0,
            "legal answers": len(legal)}

prof = pd.DataFrame({"worked solutions": op_profile([r["answer"] for r in TEST]),
                     "SFT": op_profile(SFT_OUT[:nT]), "SFT→RL": op_profile(RL_OUT[:nT])}).T
print("operators in legal answers on the 300 test puzzles (shares in %)\n")
print(prof.round(1).to_string())

## 13 · Read TensorBoard, then say only what the numbers allow

| scalar | what it should do | what it means if it doesn't |
|---|---|---|
| `train/loss` (sft run) | falls fast, then flattens | flat from step 0 ⇒ the mask or the learning rate is wrong |
| `eval/exact_match_sft` | rises well above the random-path floor | stuck near 0 ⇒ training and generation render differently |
| `train/reward` (grpo run) | climbs | flat ⇒ check `frac_reward_zero_std` |
| `train/frac_reward_zero_std` | well below 1 | near 1 ⇒ the 8 answers to a puzzle all score the same: nothing to compare |
| `train/completions/mean_length` | steady, near a worked solution | shrinking or growing fast ⇒ look at the samples |
| `eval/exact_match_rl` | rises | falling ⇒ RL is hurting; lower the learning rate |

**Claims these numbers do not support:**
- "RL taught it to search": compare **any of 8** before RL with **greedy** after. RL mostly turns answers the model could already sample into answers it commits to.
- "RL made it better at everything": check **any of 8**. Committing more usually means exploring less.
- "It learned arithmetic": the **legal** column says how often its steps are true; the **bigger** column says whether that holds for numbers it never saw.
- "RL beat SFT", when the interval crosses zero. And even when it doesn't, we didn't test spending the same 12 minutes on more SFT.